In [5]:
"""
extraccion_unificada.py

Unifica en una sola estrategia de extracción lo mejor de dos aproximaciones:

  1) Extraer por página con pymupdf4llm (page_chunks=True): da metadata de
     página y tablas ya en Markdown, pero si se chunkea página por página
     se cortan párrafos abruptamente en los saltos de página.

  2) Extraer todo el documento de una vez (to_markdown con page_chunks=False):
     no corta párrafos, pero se pierde la metadata de página y las tablas
     quedan mezcladas con el texto narrativo, sin distinguir.

Estrategia:
  - Se llama UNA sola vez a pymupdf4llm.to_markdown(..., page_chunks=True),
    que ya trae el texto markdown de cada página (tablas incluidas) y su
    número de página.
  - Las tablas se separan del texto de cada página con una regex de
    tablas Markdown (misma idea que ya tenías validada).
  - El texto narrativo (sin tablas) de todas las páginas se concatena
    nosotros mismos con un separador mínimo (un espacio, no un salto de
    párrafo), para no forzar un corte de párrafo en el borde de página.
  - Se guarda un mapa de offsets -> número de página del texto concatenado,
    así que después de chunkear con RecursiveCharacterTextSplitter cada
    chunk final recupera su(s) página(s) de origen.
  - Las tablas se entregan como Documents aparte (sin trocear), con
    tipo="tabla" y su página, tal como en tu approach 1.

Limitación conocida: si una tabla real del PDF está partida entre dos
páginas, aquí queda como dos fragmentos de tabla (uno por página), igual
que en tu approach 1 original. Fusionar tablas partidas entre páginas
requeriría heurísticas adicionales (mismo número de columnas, página N
termina en tabla y N+1 empieza en tabla, etc.) que no se implementan aquí.

Instalar dependencias:
    pip install pymupdf4llm langchain-text-splitters langchain-core
"""

import re
from typing import Dict, List, Optional, Tuple

import pymupdf4llm

try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ImportError:  # compatibilidad con instalaciones más viejas de langchain
    from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_core.documents import Document


# Detecta tablas en formato Markdown (encabezado + fila separadora +
# filas de datos). Soporta separadores con alineación tipo :---, ---:, :---:
PATRON_TABLA_MD = re.compile(
    r'(^\|.*\|\s*\n'
    r'(?:\|[ \t]*:?-+:?[ \t]*\|.*\n)'
    r'(?:\|.*\|\s*\n?)+)',
    re.MULTILINE,
)


def _separar_tablas_de_texto(texto: str, numero_pagina: Optional[int]) -> Tuple[str, List[Dict]]:
    """
    Encuentra tablas Markdown dentro del texto de una página, las saca del
    texto narrativo (dejando un marcador de referencia) y las devuelve
    aparte con su página asociada.
    """
    tablas_encontradas = PATRON_TABLA_MD.findall(texto)

    tablas = [
        {"numero_pagina": numero_pagina, "contenido_md": t.strip()}
        for t in tablas_encontradas
    ]

    def _reemplazo(match):
        idx = _reemplazo.contador
        _reemplazo.contador += 1
        return f"\n[tabla {idx + 1} de la página {numero_pagina}]\n"
    _reemplazo.contador = 0

    texto_sin_tablas = PATRON_TABLA_MD.sub(_reemplazo, texto)
    return texto_sin_tablas, tablas


def extraer_documento_unificado(
    pdf_path: str,
    table_strategy: str = "lines_strict",
    header: bool = False,
    footer: bool = False,
) -> Dict:
    """
    Extrae el PDF completo en una sola pasada y devuelve:
      - "texto_narrativo": str, todo el texto narrativo concatenado y
        continuo (sin las tablas).
      - "mapa_paginas": List[(offset_inicio, offset_fin, numero_pagina)]
        sobre "texto_narrativo".
      - "tablas": List[Dict] con "numero_pagina" y "contenido_md".
    """
    paginas = pymupdf4llm.to_markdown(
        pdf_path,
        page_chunks=True,
        table_strategy=table_strategy,
        header=header,
        footer=footer,
    )

    if isinstance(paginas, str):
        # Puede pasar si pymupdf-layout está instalado e ignora page_chunks
        # en tu versión de pymupdf4llm. Revisa versiones o desinstala
        # pymupdf-layout si te encuentras con esto.
        raise RuntimeError(
            "pymupdf4llm.to_markdown no devolvió una lista por página "
            "(devolvió texto plano). Revisa tu versión de pymupdf4llm / "
            "si tienes pymupdf-layout instalado, ya que puede cambiar "
            "el comportamiento de page_chunks."
        )

    partes_texto: List[str] = []
    mapa_paginas: List[Tuple[int, int, Optional[int]]] = []
    tablas_extraidas: List[Dict] = []
    offset_actual = 0

    for pagina in paginas:
        metadata_pagina = pagina.get("metadata", {})
        numero_pagina = metadata_pagina.get("page", metadata_pagina.get("page_number"))

        texto_pagina = pagina.get("text", "")
        texto_sin_tablas, tablas_pagina = _separar_tablas_de_texto(texto_pagina, numero_pagina)
        tablas_extraidas.extend(tablas_pagina)

        if partes_texto:
            partes_texto.append(" ")
            offset_actual += 1

        inicio = offset_actual
        partes_texto.append(texto_sin_tablas)
        offset_actual += len(texto_sin_tablas)
        fin = offset_actual

        mapa_paginas.append((inicio, fin, numero_pagina))

    for i, tabla in enumerate(tablas_extraidas):
        tabla["indice"] = i

    return {
        "texto_narrativo": "".join(partes_texto),
        "mapa_paginas": mapa_paginas,
        "tablas": tablas_extraidas,
    }


def _paginas_de_rango(mapa_paginas, inicio: int, fin: int) -> List[int]:
    """Devuelve las páginas cuyo rango de offsets se solapa con [inicio, fin)."""
    paginas = [
        num for (ini_p, fin_p, num) in mapa_paginas
        if ini_p < fin and fin_p > inicio and num is not None
    ]
    return paginas or [None]


def construir_documentos(
    pdf_path: str,
    chunk_size: int,
    chunk_overlap: int = 0,
    table_strategy: str = "lines_strict",
    header: bool = False,
    footer: bool = False,
) -> List[Document]:
    """
    Construye la lista final de Documents (langchain) para una configuración
    de RAG dada. La extracción queda fija/unificada; lo único que varía
    entre configuraciones en esta función es chunk_size (y chunk_overlap).

    Documents de tipo "texto": chunkeados sobre el documento completo y
    continuo, con metadata {"tipo": "texto", "pagina_inicio", "pagina_fin",
    "fuente", "chunk_size_config"}.

    Documents de tipo "tabla": uno por tabla, sin trocear, con metadata
    {"tipo": "tabla", "pagina", "fuente", "indice_tabla"}.
    """
    extraido = extraer_documento_unificado(
        pdf_path, table_strategy=table_strategy, header=header, footer=footer
    )
    texto_narrativo = extraido["texto_narrativo"]

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        add_start_index=True,  # langchain calcula el offset real del chunk
    )
    chunks_texto = splitter.create_documents([texto_narrativo])

    documentos: List[Document] = []

    for chunk in chunks_texto:
        inicio = chunk.metadata.get("start_index", 0)
        fin = inicio + len(chunk.page_content)
        paginas = _paginas_de_rango(extraido["mapa_paginas"], inicio, fin)
        paginas_validas = [p for p in paginas if p is not None] or [None]

        documentos.append(Document(
            page_content=chunk.page_content,
            metadata={
                "tipo": "texto",
                "pagina_inicio": min(paginas_validas) if paginas_validas[0] is not None else None,
                "pagina_fin": max(paginas_validas) if paginas_validas[0] is not None else None,
                "fuente": pdf_path,
                "chunk_size_config": chunk_size,
            },
        ))

    for tabla in extraido["tablas"]:
        documentos.append(Document(
            page_content=tabla["contenido_md"],
            metadata={
                "tipo": "tabla",
                "pagina": tabla["numero_pagina"],
                "fuente": pdf_path,
                "indice_tabla": tabla["indice"],
            },
        ))

    return documentos


def resumen_documentos(documentos: List[Document]) -> None:
    """Impresión rápida para sanity-check al probar una configuración."""
    textos = [d for d in documentos if d.metadata.get("tipo") == "texto"]
    tablas = [d for d in documentos if d.metadata.get("tipo") == "tabla"]
    largos = [len(d.page_content) for d in textos]

    print(f"Chunks de texto: {len(textos)}")
    if largos:
        print(f"  largo promedio: {sum(largos)/len(largos):.0f} caracteres")
        print(f"  largo min/max:  {min(largos)} / {max(largos)}")
    print(f"Tablas extraídas: {len(tablas)}")


if __name__ == "__main__":
    # Ejemplo de uso: la extracción queda fija; lo que varía en tus
    # "configuraciones" es chunk_size (y luego, aparte, el embedding model).
    pdf = "./factor--1.pdf"
    chunk_sizes = [200, 500, 1200]

    for cs in chunk_sizes:
        print(f"\n--- chunk_size={cs} ---")
        docs = construir_documentos(pdf, chunk_size=cs, chunk_overlap=int(cs * 0.15))
        resumen_documentos(docs)


--- chunk_size=200 ---
Chunks de texto: 297
  largo promedio: 162 caracteres
  largo min/max:  25 / 199
Tablas extraídas: 10

--- chunk_size=500 ---
Chunks de texto: 130
  largo promedio: 364 caracteres
  largo min/max:  38 / 499
Tablas extraídas: 10

--- chunk_size=1200 ---
Chunks de texto: 51
  largo promedio: 904 caracteres
  largo min/max:  40 / 1199
Tablas extraídas: 10


In [4]:
docs

[Document(metadata={'tipo': 'texto', 'pagina_inicio': 1, 'pagina_fin': 1, 'fuente': './factor--1.pdf', 'chunk_size_config': 1500}, page_content='# **5. Resultados de la Autoevaluación** \n\nEste capítulo presenta el análisis por cada factor. Al inicio de cada uno, se enumeran las conclusiones del anterior proceso de autoevaluación (2012-2017) al mismo tiempo que se hace una actualización de lo mencionado en cada una para evidenciar los avances o principales logros obtenidos en el período 2018-2023, permitiendo así la comparación de ambos procesos. Es importante anotar que el proceso pasado de acreditación conjunta CNA-ARCUSUR siguió de forma sistemática las directrices contempladas en los documentos: Lineamientos para la acreditación de programas académicos de pregrado, CNA, 2013, Guía de procedimiento No. 3 para la autoevaluación con fines de acreditación de programas de pregrado, CNA, 2013 y el Sistema de Información SACES-CNA. Una vez terminado el proceso para producir el informe de